<a href="https://colab.research.google.com/github/dtc-oss/python_hw/blob/hw_5/hw5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задание 5. Polars, pandas и оптимизация типов

Три части задания, датасеты `train.csv` и `Housing.csv` должны лежать рядом с ноутбуком.
1. Анализ в Polars (`train.csv`).
2. Вычисления в pandas с bottleneck (`train.csv`)
3. Оптимизация памяти таблицы (`Housing.csv`).

In [1]:
from pathlib import Path

def find_dataset(filename: str) -> Path:
    """Ищет файл рядом с ноутбуком/в текущей папке и в стандартной папке задания."""
    candidate = [
        Path(filename),
        Path.cwd() / filename,
        Path.cwd().parent / filename,
    ]
    for path in candidate:
        if path.is_file():
            return path.resolve()
    searched = '\\n'.join(f' - {path}' for path in candidate)
    raise FileNotFoundError(
        f'Не найден файл {filename}. Поместите его рядом с ноутбуком. Проверены пути:\\n{searched}'
    )

TITANIC_PATH = find_dataset("train.csv")
HOUSING_PATH = find_dataset("Housing.csv")
print(TITANIC_PATH)
print(HOUSING_PATH)

/content/train.csv
/content/Housing.csv


## 1. Polars: анализ Titanic
### Загрузка и общая информация

In [2]:
import polars as pl

titanic = pl.read_csv(TITANIC_PATH)
print(f"Размер: {titanic.height} строк, {titanic.width} столбцов")
titanic.head()

Размер: 891 строк, 12 столбцов


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


In [3]:
# Тип каждого столбца
for column, dtype in titanic.schema.items():
    print(f"{column}: {dtype}")

# Число пропусков и их доля
missing = titanic.null_count().transpose(
    include_header=True, header_name="Столбец",
    column_names=["Пропусков"]
)
missing = missing.with_columns(
    (pl.col("Пропусков") / titanic.height * 100).round(2).alias("Пропусков, %")
)
display(missing)

# Средние значения числовых столбцов
means = titanic.select(pl.col(pl.Int64, pl.Float64).mean())
display(means.transpose(
    include_header=True, header_name="Столбец",
    column_names=["Среднее"]
))

# Сводная статистика
display(titanic.describe())

PassengerId: Int64
Survived: Int64
Pclass: Int64
Name: String
Sex: String
Age: Float64
SibSp: Int64
Parch: Int64
Ticket: String
Fare: Float64
Cabin: String
Embarked: String


Столбец,Пропусков,"Пропусков, %"
str,u32,f64
"""PassengerId""",0,0.0
"""Survived""",0,0.0
"""Pclass""",0,0.0
"""Name""",0,0.0
"""Sex""",0,0.0
…,…,…
"""Parch""",0,0.0
"""Ticket""",0,0.0
"""Fare""",0,0.0


Столбец,Среднее
str,f64
"""PassengerId""",446.0
"""Survived""",0.383838
"""Pclass""",2.308642
"""Age""",29.699118
"""SibSp""",0.523008
"""Parch""",0.381594
"""Fare""",32.204208


statistic,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""


### Количество пассажиров каждого класса

In [4]:
class_counts = (
    titanic.get_column("Pclass")
    .value_counts()
    .sort("Pclass")
    .rename({"count": "Количество пассажиров"})
)
display(class_counts)

Pclass,Количество пассажиров
i64,u32
1,216
2,184
3,491


### Количество выживших мужчин и женщин на корабле

In [5]:
# Survived равен 1 для выживших и 0 для погибших, поэтому сумма даёт число выживших.
survival = (
    titanic.group_by("Sex")
    .agg(
        pl.col("Survived").sum().alias("Выжило"),
        pl.len().alias("Всего пассажиров")
    )
    .with_columns(
        (pl.col("Выжило") / pl.col("Всего пассажиров") * 100)
        .round(2).alias("Выжило, %")
    )
    .sort("Выжило", descending=True)
)
display(survival)

Sex,Выжило,Всего пассажиров,"Выжило, %"
str,i64,u32,f64
"""female""",233,314,74.2
"""male""",109,577,18.89


### Пассажиры старше 44 лет

In [6]:
older = titanic.filter(pl.col("Age") > 44)
print(f"Количество пассажиров старше 44 лет: {older.height}")
display(older.head(10))

Количество пассажиров старше 44 лет: 115


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
7,0,1,"""McCarthy, Mr. Timothy J""","""male""",54.0,0,0,"""17463""",51.8625,"""E46""","""S"""
12,1,1,"""Bonnell, Miss. Elizabeth""","""female""",58.0,0,0,"""113783""",26.55,"""C103""","""S"""
16,1,2,"""Hewlett, Mrs. (Mary D Kingcome…","""female""",55.0,0,0,"""248706""",16.0,null,"""S"""
34,0,2,"""Wheadon, Mr. Edward H""","""male""",66.0,0,0,"""C.A. 24579""",10.5,null,"""S"""
53,1,1,"""Harper, Mrs. Henry Sleeper (My…","""female""",49.0,1,0,"""PC 17572""",76.7292,"""D33""","""C"""
55,0,1,"""Ostby, Mr. Engelhart Cornelius""","""male""",65.0,0,1,"""113509""",61.9792,"""B30""","""C"""
63,0,1,"""Harris, Mr. Henry Birkhardt""","""male""",45.0,1,0,"""36973""",83.475,"""C83""","""S"""
93,0,1,"""Chaffee, Mr. Herbert Fuller""","""male""",46.0,1,0,"""W.E.P. 5734""",61.175,"""E31""","""S"""
95,0,3,"""Coxon, Mr. Daniel""","""male""",59.0,0,0,"""364500""",7.25,null,"""S"""


## 2. Ускорение вычислений в pandas
### Средний возраст и стандартное отклонение

In [7]:
import numpy as np
import pandas as pd
import bottleneck as bn

df = pd.read_csv(TITANIC_PATH)
ages = df["Age"].to_numpy()

print(f"Средний возраст: {bn.nanmean(ages):.4f}")
print(f"Стандартное отклонение: {bn.nanstd(ages, ddof=1):.4f}")
print(f"Заполненных значений возраста: {np.count_nonzero(~np.isnan(ages))} из {len(ages)}")

# ddof=1 даёт выборочное стандартное отклонение, как pandas.Series.std().
print(f"bottleneck ddof=0: {bn.nanstd(ages):.4f}")
print(f"bottleneck ddof=1: {bn.nanstd(ages, ddof=1):.4f}")
print(f"pandas std():      {df['Age'].std():.4f}")
print("Результаты совпадают с pandas:", np.isclose(bn.nanstd(ages, ddof=1), df["Age"].std()))

Средний возраст: 29.6991
Стандартное отклонение: 14.5265
Заполненных значений возраста: 714 из 891
bottleneck ddof=0: 14.5163
bottleneck ddof=1: 14.5265
pandas std():      14.5265
Результаты совпадают с pandas: True


### Новый столбец `Fare_new = Fare × 1.3`

In [8]:
df['Fare_new'] = df['Fare'].apply(lambda fare: fare * 1.3)
display(df[['PassengerId', 'Fare', 'Fare_new']].head(10))
print('Проверка вычисления:', np.allclose(df['Fare_new'], df['Fare'] * 1.3))

,PassengerId,Fare,Fare_new
0,1,7.2500,9.42500
1,2,71.2833,92.66829
2,3,7.9250,10.30250
3,4,53.1000,69.03000
4,5,8.0500,10.46500
5,6,8.4583,10.99579
6,7,51.8625,67.42125
7,8,21.0750,27.39750
8,9,11.1333,14.47329
9,10,30.0708,39.09204


Проверка вычисления: True


## 3. Оптимизация типов pandas
### Загрузка Housing и просмотр данных

In [9]:
housing = pd.read_csv(HOUSING_PATH)
housing_before = housing.copy(deep=True)

print(f'Размер Housing: {housing.shape[0]} строк × {housing.shape[1]} столбцов')
display(housing.head())
print('Типы исходных данных:')
display(housing.dtypes.rename('dtype').to_frame())

Размер Housing: 545 строк × 13 столбцов


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


Типы исходных данных:


,dtype
price,int64
area,int64
bedrooms,int64
bathrooms,int64
stories,int64
mainroad,object
guestroom,object
basement,object
hotwaterheating,object
airconditioning,object


### Обоснование выбранных типов данных

- `price`: целое значение; диапазон значений позволяет использовать `int32` вместо `int64`.
- `area`: целое значение, максимальная площадь помещается в `int16`.
- `bedrooms`, `bathrooms`, `stories`, `parking`: небольшие неотрицательные целые числа, для которых достаточно `int8`.
- `mainroad`, `guestroom`, `basement`, `hotwaterheating`, `airconditioning`, `prefarea`: бинарные категории `yes/no`; тип `category` не дублирует длинные строки в каждой строке.
- `furnishingstatus`: всего несколько повторяющихся категорий, поэтому `category` экономичнее типа `object`.

In [10]:
housing_opt = housing.copy(deep=True)

# Целочисленные столбцы: downcast выбирает меньший безопасный целый тип.
for column in ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'parking']:
    housing_opt[column] = pd.to_numeric(housing_opt[column], downcast='integer')

# Категориальные столбцы с малым числом повторяющихся значений.
category_columns = [
    'mainroad', 'guestroom', 'basement', 'hotwaterheating',
    'airconditioning', 'prefarea', 'furnishingstatus'
]
for column in category_columns:
    housing_opt[column] = housing_opt[column].astype('category')

# Проверяем, что данные не изменились из-за оптимизации.
pd.testing.assert_frame_equal(
    housing_before,
    housing_opt.astype({c: housing_before[c].dtype for c in category_columns}),
    check_dtype=False,
)

memory_before = int(housing_before.memory_usage(index=True, deep=True).sum())
memory_after = int(housing_opt.memory_usage(index=True, deep=True).sum())
reduction_percent = (memory_before - memory_after) / memory_before * 100

dtype_comparison = pd.DataFrame({
    'dtype_before': housing_before.dtypes.astype(str),
    'dtype_after': housing_opt.dtypes.astype(str),
    'memory_before_bytes': housing_before.memory_usage(index=False, deep=True),
    'memory_after_bytes': housing_opt.memory_usage(index=False, deep=True),
})
display(dtype_comparison)
print(f'Потребление памяти до оптимизации: {memory_before:,} байт ({memory_before / 1024:.2f} KiB)')
print(f'Потребление памяти после оптимизации: {memory_after:,} байт ({memory_after / 1024:.2f} KiB)')
print(f'Снижение памяти: {reduction_percent:.2f}%')
print('\nИтоговые типы:')
display(housing_opt.dtypes.rename('optimized_dtype').to_frame())

,dtype_before,dtype_after,memory_before_bytes,memory_after_bytes
price,int64,int32,4360,2180
area,int64,int16,4360,1090
bedrooms,int64,int8,4360,545
bathrooms,int64,int8,4360,545
stories,int64,int8,4360,545
mainroad,object,category,28263,756
guestroom,object,category,27892,756
basement,object,category,27986,756
hotwaterheating,object,category,27820,756
airconditioning,object,category,27967,756


Потребление памяти до оптимизации: 227,244 байт (221.92 KiB)
Потребление памяти после оптимизации: 10,952 байт (10.70 KiB)
Снижение памяти: 95.18%

Итоговые типы:


,optimized_dtype
price,int32
area,int16
bedrooms,int8
bathrooms,int8
stories,int8
mainroad,category
guestroom,category
basement,category
hotwaterheating,category
airconditioning,category
